In [9]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
import kagglehub
import pandas as pd
import tensorflow as tf

# Télécharger le dataset
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Path:", path)

# Charger les données
df = pd.read_csv(path + "/spam.csv", encoding='latin-1')
print(df.head())

Path: C:\Users\Pc ForYou\.cache\kagglehub\datasets\uciml\sms-spam-collection-dataset\versions\1
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [2]:
# Voir la taille du dataset
print("Nombre de messages :", len(df))

# Voir toutes les colonnes
print("\nColonnes :", df.columns.tolist())

# Voir la répartition Ham vs Spam
print("\nRépartition des labels :")
print(df['v1'].value_counts())

Nombre de messages : 5572

Colonnes : ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

Répartition des labels :
v1
ham     4825
spam     747
Name: count, dtype: int64


In [3]:
# Garder uniquement les colonnes utiles et les renommer
df = df[['v1', 'v2']]
df.columns = ['label', 'texte']

# Convertir ham → 0, spam → 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Vérifier qu'il n'y a pas de valeurs manquantes
print("Valeurs manquantes :\n", df.isnull().sum())

# Aperçu final
print("\n", df.head(10))

Valeurs manquantes :
 label    0
texte    0
dtype: int64

    label                                              texte
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
5      1  FreeMsg Hey there darling it's been 3 week's n...
6      0  Even my brother is not like to speak with me. ...
7      0  As per your request 'Melle Melle (Oru Minnamin...
8      1  WINNER!! As a valued network customer you have...
9      1  Had your mobile 11 months or more? U R entitle...


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df['texte'],   # les messages
    df['label'],   # les labels (0 ou 1)
    test_size=0.2, # 20% pour la validation
    random_state=42
)

print("Taille entraînement :", len(X_train))
print("Taille validation   :", len(X_val))

Taille entraînement : 4457
Taille validation   : 1115


In [5]:
import tensorflow as tf

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Créer les datasets TensorFlow
train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train.values, y_train.values)
).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_val.values, y_val.values)
).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

print("✅ Datasets TensorFlow prêts !")

✅ Datasets TensorFlow prêts !


In [6]:


#Préparation de la couche de vectorisation du texte
VOCAB_SIZE = 10000          # Nombre max de mots uniques à retenir
MAX_SEQUENCE_LENGTH = 100   # Longueur max d'un message (on coupe ou on comble si c'est plus court)

encoder = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=MAX_SEQUENCE_LENGTH
)

# Adapter l'encodeur au vocabulaire de nos données d'entraînement
encoder.adapt(X_train.values)

# Construction de l'architecture du modèle
model_maison = tf.keras.Sequential([
    # Entrée : prend une chaîne de caractères (le texte brut)
    tf.keras.Input(shape=(1,), dtype=tf.string),
    
    # Étape d'encodage textuel numérique
    encoder,
    
    # Couche d'Embedding (transforme chaque mot en un vecteur dense de 64 chiffres)
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=64, mask_zero=True),
    
    # Couche Récurrente Bi-LSTM (analyse le texte de gauche à droite ET de droite à gauche)
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    
    # Couches d'ajustement et régularisation (évite le surapprentissage)
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.5), 
    
    # Couche de sortie : 1 neurone avec Sigmoid pour une décision binaire (0 ou 1)
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compilation du modèle
model_maison.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

print("Modèle fait maison construit avec succès !")
model_maison.summary()

Modèle fait maison construit avec succès !


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 666,945 (2.54 MB)

 Trainable params: 666,945 (2.54 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Étape 4 — Entraînement et Sauvegarde des Poids (Zéro bug Windows possible)
print("Début de l'entraînement...")

history_maison = model_maison.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds,
    verbose=1
)

try:
    # On tente la sauvegarde complète classique
    model_maison.save("modele_maison_texte.keras")
    print("\n✅ Modèle complet sauvegardé sous 'modele_maison_texte.keras' !")
except Exception:
    # Si Windows bloque à cause des caractères bizarres, on active la sécurité :
    print("\n⚠️ Conflit de caractères Windows détecté.")
    print("-> Activation du plan de secours : Sauvegarde des poids numériques.")
    
    # On sauvegarde uniquement les poids (les chiffres des neurones)
    model_maison.save_weights("modele_maison_poids.weights.h5")
    print("✅ Poids du modèle sauvegardés avec succès sous 'modele_maison_poids.weights.h5' ! (Zéro bug)")

Début de l'entraînement...
Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 79ms/step - accuracy: 1.0000 - loss: 2.8259e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9821 - val_loss: 0.2014 - val_precision: 0.9514 - val_recall: 0.9133
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 15s 108ms/step - accuracy: 1.0000 - loss: 2.5572e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9830 - val_loss: 0.2066 - val_precision: 0.9580 - val_recall: 0.9133
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 12s 88ms/step - accuracy: 1.0000 - loss: 2.5855e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9821 - val_loss: 0.2108 - val_precision: 0.9514 - val_recall: 0.9133
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 13s 93ms/step - accuracy: 1.0000 - loss: 2.2339e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9821 - val_loss: 0.2139 - val_precision: 0.9514 - val_recall: 0.9133
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 13s 94ms/step - accuracy: 1.0000 - loss: 2.3452e-04 - precision: 1.0000 -